# Mini-Project 2: FastAPI ML Prediction Service

**Goal:** Build, test, and load-test a production-ready FastAPI service that serves churn predictions.

**Estimated time:** 2-3 hours

**What you'll practice:**
- FastAPI endpoints (GET, POST, batch)
- Pydantic validation and schema design
- TestClient testing (unit & integration)
- Middleware (timing, counters, metrics)
- Load testing with concurrent requests

In [ ]:
import time
import json
import pickle
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from typing import List, Optional

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("All imports successful!")

---
## Part 1: Train and Save Model

We'll generate a synthetic churn dataset, train a GradientBoostingClassifier, evaluate it,
and keep the serialized model in memory for our FastAPI service.

In [ ]:
# Generate synthetic churn dataset
np.random.seed(42)
n_samples = 5000

data = {
    'tenure_months': np.random.randint(0, 72, n_samples),
    'monthly_charges': np.round(np.random.uniform(20, 120, n_samples), 2),
    'contract': np.random.choice([0, 1, 2], n_samples, p=[0.5, 0.25, 0.25]),  # 0=Month-to-month, 1=One year, 2=Two year
    'internet_service': np.random.choice([0, 1, 2], n_samples, p=[0.35, 0.35, 0.30]),  # 0=DSL, 1=Fiber optic, 2=No
    'online_security': np.random.choice([0, 1], n_samples, p=[0.5, 0.5]),
    'tech_support': np.random.choice([0, 1], n_samples, p=[0.5, 0.5]),
}

data['total_charges'] = np.round(data['tenure_months'] * data['monthly_charges'] * np.random.uniform(0.8, 1.2, n_samples), 2)

# Create churn label with realistic logic
churn_prob = (
    0.3
    - 0.004 * data['tenure_months']
    + 0.003 * data['monthly_charges']
    - 0.15 * (data['contract'] == 2).astype(float)
    - 0.10 * (data['contract'] == 1).astype(float)
    + 0.10 * (data['internet_service'] == 1).astype(float)
    - 0.08 * data['online_security']
    - 0.06 * data['tech_support']
    + np.random.normal(0, 0.1, n_samples)
)
churn_prob = np.clip(churn_prob, 0.01, 0.99)
data['churn'] = (np.random.random(n_samples) < churn_prob).astype(int)

df = pd.DataFrame(data)
print(f"Dataset shape: {df.shape}")
print(f"Churn rate: {df['churn'].mean():.2%}")
print()
df.head()

In [ ]:
# Train / test split
FEATURE_NAMES = ['tenure_months', 'monthly_charges', 'contract', 'internet_service',
                 'online_security', 'tech_support', 'total_charges']

X = df[FEATURE_NAMES].values
y = df['churn'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train model
model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

In [ ]:
# Save model to in-memory buffer (no file system dependency)
model_buffer = io.BytesIO()
pickle.dump(model, model_buffer)
model_buffer.seek(0)

# Model metadata
MODEL_METADATA = {
    'model_version': '1.0.0',
    'feature_names': FEATURE_NAMES,
    'model_type': 'GradientBoostingClassifier',
    'n_estimators': 100,
    'training_accuracy': float(accuracy_score(y_test, y_pred)),
    'training_f1': float(f1_score(y_test, y_pred)),
    'training_samples': len(X_train),
}

# Reload from buffer to verify serialization works
loaded_model = pickle.load(model_buffer)
assert np.array_equal(loaded_model.predict(X_test[:5]), model.predict(X_test[:5]))

print("Model serialized and verified successfully!")
print(f"Model metadata: {json.dumps(MODEL_METADATA, indent=2)}")

---
## Part 2: Define Pydantic Schemas

Design request/response schemas with Pydantic validation. These schemas enforce
data quality at the API boundary before data reaches the model.

In [ ]:
# Mapping dicts for categorical encoding
CONTRACT_MAP = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
INTERNET_MAP = {'DSL': 0, 'Fiber optic': 1, 'No': 2}

# TODO: Create CustomerFeatures Pydantic model
# Fields:
#   tenure_months: int, Field(ge=0, le=120, description="Months as customer")
#   monthly_charges: float, Field(ge=0, le=500, description="Monthly bill amount")
#   contract: str - one of 'Month-to-month', 'One year', 'Two year'
#   internet_service: str - one of 'DSL', 'Fiber optic', 'No'
#   online_security: int (0 or 1)
#   tech_support: int (0 or 1)
#   total_charges: float, Field(ge=0, description="Total charges to date")
#
# Add a method to_feature_array() that returns a numpy array in the correct order

# YOUR CODE HERE


# --- Commented Solution ---
# class CustomerFeatures(BaseModel):
#     tenure_months: int = Field(ge=0, le=120, description="Months as customer")
#     monthly_charges: float = Field(ge=0, le=500, description="Monthly bill amount")
#     contract: str = Field(description="Contract type: Month-to-month, One year, or Two year")
#     internet_service: str = Field(description="Internet service type: DSL, Fiber optic, or No")
#     online_security: int = Field(ge=0, le=1, description="Has online security (0 or 1)")
#     tech_support: int = Field(ge=0, le=1, description="Has tech support (0 or 1)")
#     total_charges: float = Field(ge=0, description="Total charges to date")
#
#     def to_feature_array(self) -> np.ndarray:
#         """Convert to numpy array matching model feature order."""
#         return np.array([[
#             self.tenure_months,
#             self.monthly_charges,
#             CONTRACT_MAP[self.contract],
#             INTERNET_MAP[self.internet_service],
#             self.online_security,
#             self.tech_support,
#             self.total_charges,
#         ]])

In [ ]:
# TODO: Create PredictionResponse Pydantic model
# Fields:
#   customer_id: Optional[str] = None
#   churn_probability: float
#   churn_prediction: bool
#   model_version: str

# YOUR CODE HERE


# --- Commented Solution ---
# class PredictionResponse(BaseModel):
#     customer_id: Optional[str] = None
#     churn_probability: float
#     churn_prediction: bool
#     model_version: str

In [ ]:
# TODO: Create BatchRequest and BatchResponse models
# BatchRequest:
#   customers: List[CustomerFeatures]
# BatchResponse:
#   predictions: List[PredictionResponse]
#   batch_size: int
#   processing_time_ms: float

# YOUR CODE HERE


# --- Commented Solution ---
# class BatchRequest(BaseModel):
#     customers: List[CustomerFeatures]
#
# class BatchResponse(BaseModel):
#     predictions: List[PredictionResponse]
#     batch_size: int
#     processing_time_ms: float

In [ ]:
# Quick validation test
sample_customer = CustomerFeatures(
    tenure_months=24,
    monthly_charges=65.50,
    contract='One year',
    internet_service='Fiber optic',
    online_security=1,
    tech_support=0,
    total_charges=1572.00
)
print(f"Sample customer: {sample_customer}")
print(f"Feature array: {sample_customer.to_feature_array()}")
print(f"Feature array shape: {sample_customer.to_feature_array().shape}")

---
## Part 3: Build FastAPI App

Create a FastAPI application with health check, single prediction, and batch prediction endpoints.

In [ ]:
# TODO: Build the FastAPI app
#
# 1. Create FastAPI app with:
#    - title="Churn Prediction Service"
#    - description="Production-ready ML prediction service for customer churn"
#    - version="1.0.0"
#
# 2. GET /health endpoint:
#    - Return {"status": "healthy", "model_loaded": True, "model_version": MODEL_METADATA['model_version']}
#
# 3. POST /predict endpoint:
#    - Accept CustomerFeatures
#    - Validate contract value is in CONTRACT_MAP
#    - Validate internet_service value is in INTERNET_MAP
#    - Use loaded_model to predict
#    - Return PredictionResponse
#
# 4. POST /predict/batch endpoint:
#    - Accept BatchRequest
#    - Return BatchResponse with predictions for all customers
#    - Include processing time

# YOUR CODE HERE


# --- Commented Solution ---
# app = FastAPI(
#     title="Churn Prediction Service",
#     description="Production-ready ML prediction service for customer churn",
#     version="1.0.0"
# )
#
# @app.get("/health")
# def health_check():
#     return {
#         "status": "healthy",
#         "model_loaded": True,
#         "model_version": MODEL_METADATA['model_version']
#     }
#
# @app.post("/predict", response_model=PredictionResponse)
# def predict(customer: CustomerFeatures):
#     # Validate categorical fields
#     if customer.contract not in CONTRACT_MAP:
#         raise HTTPException(
#             status_code=422,
#             detail=f"Invalid contract type: {customer.contract}. Must be one of {list(CONTRACT_MAP.keys())}"
#         )
#     if customer.internet_service not in INTERNET_MAP:
#         raise HTTPException(
#             status_code=422,
#             detail=f"Invalid internet service: {customer.internet_service}. Must be one of {list(INTERNET_MAP.keys())}"
#         )
#
#     features = customer.to_feature_array()
#     probability = float(loaded_model.predict_proba(features)[0, 1])
#     prediction = bool(probability >= 0.5)
#
#     return PredictionResponse(
#         churn_probability=round(probability, 4),
#         churn_prediction=prediction,
#         model_version=MODEL_METADATA['model_version']
#     )
#
# @app.post("/predict/batch", response_model=BatchResponse)
# def predict_batch(batch: BatchRequest):
#     start_time = time.time()
#
#     if len(batch.customers) == 0:
#         raise HTTPException(status_code=422, detail="Batch cannot be empty")
#
#     predictions = []
#     for i, customer in enumerate(batch.customers):
#         if customer.contract not in CONTRACT_MAP:
#             raise HTTPException(status_code=422, detail=f"Customer {i}: Invalid contract type")
#         if customer.internet_service not in INTERNET_MAP:
#             raise HTTPException(status_code=422, detail=f"Customer {i}: Invalid internet service")
#
#         features = customer.to_feature_array()
#         probability = float(loaded_model.predict_proba(features)[0, 1])
#         prediction = bool(probability >= 0.5)
#
#         predictions.append(PredictionResponse(
#             customer_id=f"customer_{i}",
#             churn_probability=round(probability, 4),
#             churn_prediction=prediction,
#             model_version=MODEL_METADATA['model_version']
#         ))
#
#     processing_time = (time.time() - start_time) * 1000
#
#     return BatchResponse(
#         predictions=predictions,
#         batch_size=len(predictions),
#         processing_time_ms=round(processing_time, 2)
#     )

print("FastAPI app created!")
print(f"App title: {app.title}")
print(f"Routes: {[route.path for route in app.routes]}")

---
## Part 4: Test with TestClient

Use FastAPI's TestClient to write integration tests for every endpoint without starting a real server.

In [ ]:
# TODO: Create TestClient and write test functions
#
# 1. Create: client = TestClient(app)
#
# 2. test_health():
#    - GET /health
#    - Assert status_code == 200
#    - Assert response has 'status', 'model_loaded', 'model_version' keys
#    - Assert status == 'healthy'
#
# 3. test_predict_valid():
#    - POST /predict with valid customer JSON
#    - Assert status_code == 200
#    - Assert response has 'churn_probability', 'churn_prediction', 'model_version'
#    - Assert churn_probability is between 0 and 1
#
# 4. test_predict_invalid():
#    - POST /predict with tenure_months=-5 (invalid)
#    - Assert status_code == 422
#
# 5. test_predict_batch():
#    - POST /predict/batch with 5 customers
#    - Assert status_code == 200
#    - Assert len(predictions) == 5
#    - Assert batch_size == 5

# YOUR CODE HERE


# --- Commented Solution ---
# client = TestClient(app)
#
# def test_health():
#     response = client.get("/health")
#     assert response.status_code == 200
#     data = response.json()
#     assert 'status' in data
#     assert 'model_loaded' in data
#     assert 'model_version' in data
#     assert data['status'] == 'healthy'
#     assert data['model_loaded'] is True
#     print("test_health PASSED")
#
# def test_predict_valid():
#     payload = {
#         "tenure_months": 24,
#         "monthly_charges": 65.50,
#         "contract": "One year",
#         "internet_service": "Fiber optic",
#         "online_security": 1,
#         "tech_support": 0,
#         "total_charges": 1572.00
#     }
#     response = client.post("/predict", json=payload)
#     assert response.status_code == 200
#     data = response.json()
#     assert 'churn_probability' in data
#     assert 'churn_prediction' in data
#     assert 'model_version' in data
#     assert 0 <= data['churn_probability'] <= 1
#     print(f"test_predict_valid PASSED - churn_prob={data['churn_probability']:.4f}")
#
# def test_predict_invalid():
#     payload = {
#         "tenure_months": -5,
#         "monthly_charges": 65.50,
#         "contract": "One year",
#         "internet_service": "Fiber optic",
#         "online_security": 1,
#         "tech_support": 0,
#         "total_charges": 1572.00
#     }
#     response = client.post("/predict", json=payload)
#     assert response.status_code == 422
#     print("test_predict_invalid PASSED - got 422 as expected")
#
# def test_predict_batch():
#     customers = []
#     for i in range(5):
#         customers.append({
#             "tenure_months": 10 + i * 10,
#             "monthly_charges": 40.0 + i * 15,
#             "contract": ["Month-to-month", "One year", "Two year"][i % 3],
#             "internet_service": ["DSL", "Fiber optic", "No"][i % 3],
#             "online_security": i % 2,
#             "tech_support": (i + 1) % 2,
#             "total_charges": (10 + i * 10) * (40.0 + i * 15)
#         })
#     response = client.post("/predict/batch", json={"customers": customers})
#     assert response.status_code == 200
#     data = response.json()
#     assert len(data['predictions']) == 5
#     assert data['batch_size'] == 5
#     print(f"test_predict_batch PASSED - batch_size={data['batch_size']}, time={data['processing_time_ms']:.2f}ms")
#
# # Run all tests
# test_health()
# test_predict_valid()
# test_predict_invalid()
# test_predict_batch()
# print("\nAll tests passed!")

---
## Part 5: Error Handling

Test edge cases and observe how FastAPI + Pydantic auto-generate meaningful error responses.

In [ ]:
# TODO: Test error handling edge cases
#
# 1. test_missing_fields() - POST /predict with missing required fields
# 2. test_wrong_types() - POST /predict with string where int expected
# 3. test_empty_batch() - POST /predict/batch with empty customers list
# 4. test_invalid_contract() - POST /predict with contract="Invalid"
#
# For each test, print the status code and error detail to see the error format.

# YOUR CODE HERE


# --- Commented Solution ---
# def test_missing_fields():
#     """Missing required fields should return 422."""
#     payload = {"tenure_months": 24}  # Missing most fields
#     response = client.post("/predict", json=payload)
#     assert response.status_code == 422
#     errors = response.json()['detail']
#     print(f"test_missing_fields PASSED - {len(errors)} validation errors")
#     for err in errors:
#         print(f"  - {err['loc']}: {err['msg']}")
#
# def test_wrong_types():
#     """Wrong data types should return 422."""
#     payload = {
#         "tenure_months": "not_a_number",
#         "monthly_charges": 65.50,
#         "contract": "One year",
#         "internet_service": "DSL",
#         "online_security": 1,
#         "tech_support": 0,
#         "total_charges": 1572.00
#     }
#     response = client.post("/predict", json=payload)
#     assert response.status_code == 422
#     print(f"test_wrong_types PASSED - status {response.status_code}")
#
# def test_empty_batch():
#     """Empty batch should return 422."""
#     response = client.post("/predict/batch", json={"customers": []})
#     assert response.status_code == 422
#     print(f"test_empty_batch PASSED - status {response.status_code}")
#
# def test_invalid_contract():
#     """Invalid contract value should return 422."""
#     payload = {
#         "tenure_months": 24,
#         "monthly_charges": 65.50,
#         "contract": "Invalid",
#         "internet_service": "DSL",
#         "online_security": 1,
#         "tech_support": 0,
#         "total_charges": 1572.00
#     }
#     response = client.post("/predict", json=payload)
#     assert response.status_code == 422
#     print(f"test_invalid_contract PASSED - detail: {response.json()['detail']}")
#
# test_missing_fields()
# test_wrong_types()
# test_empty_batch()
# test_invalid_contract()
# print("\nAll error handling tests passed!")

---
## Part 6: Middleware

Add middleware for request timing, request counting, and a metrics endpoint to monitor service performance.

In [ ]:
# TODO: Add middleware to the app
#
# 1. Create a request_stats dict to track:
#    - total_requests: int
#    - total_latency: float
#    - latencies: list of floats
#
# 2. Add timing middleware using @app.middleware("http"):
#    - Record start time
#    - Call the endpoint (response = await call_next(request))
#    - Record end time
#    - Update request_stats
#    - Add X-Process-Time header to response
#
# 3. Add GET /metrics endpoint:
#    - Return total_requests, avg_latency_ms, p50/p95/p99 latency
#
# 4. Rebuild TestClient and verify middleware works

# YOUR CODE HERE


# --- Commented Solution ---
# # We need to rebuild the app with middleware
# # (In a real project, middleware would be added before starting the server)
#
# request_stats = {
#     'total_requests': 0,
#     'total_latency': 0.0,
#     'latencies': []
# }
#
# @app.middleware("http")
# async def timing_middleware(request, call_next):
#     start_time = time.time()
#     response = await call_next(request)
#     process_time = time.time() - start_time
#
#     # Update stats
#     request_stats['total_requests'] += 1
#     request_stats['total_latency'] += process_time
#     request_stats['latencies'].append(process_time * 1000)  # ms
#
#     # Add header
#     response.headers['X-Process-Time'] = f"{process_time:.4f}"
#     return response
#
# @app.get("/metrics")
# def get_metrics():
#     if request_stats['total_requests'] == 0:
#         return {"total_requests": 0, "message": "No requests yet"}
#
#     latencies = request_stats['latencies']
#     return {
#         "total_requests": request_stats['total_requests'],
#         "avg_latency_ms": round(np.mean(latencies), 2),
#         "p50_latency_ms": round(np.percentile(latencies, 50), 2),
#         "p95_latency_ms": round(np.percentile(latencies, 95), 2),
#         "p99_latency_ms": round(np.percentile(latencies, 99), 2),
#     }
#
# # Rebuild client with updated app
# client = TestClient(app)
# print("Middleware added. Routes:", [route.path for route in app.routes])

In [ ]:
# Test middleware
# Reset stats
request_stats['total_requests'] = 0
request_stats['total_latency'] = 0.0
request_stats['latencies'] = []

# Make a few requests
for _ in range(10):
    client.get("/health")

payload = {
    "tenure_months": 24,
    "monthly_charges": 65.50,
    "contract": "One year",
    "internet_service": "Fiber optic",
    "online_security": 1,
    "tech_support": 0,
    "total_charges": 1572.00
}
for _ in range(10):
    client.post("/predict", json=payload)

# Check metrics
metrics_response = client.get("/metrics")
print("Metrics after 20 requests (+ 1 metrics request):")
print(json.dumps(metrics_response.json(), indent=2))

---
## Part 7: Load Test

Simulate concurrent traffic to measure throughput and latency under load using ThreadPoolExecutor.

In [ ]:
# TODO: Create load_test function
#
# def load_test(client, n_requests=200, n_workers=10):
#     1. Reset request_stats
#     2. Create a sample payload
#     3. Define send_request() that:
#        - Records start time
#        - POSTs to /predict
#        - Records end time
#        - Returns (latency_ms, status_code)
#     4. Use ThreadPoolExecutor(max_workers=n_workers) to submit n_requests
#     5. Collect results with as_completed
#     6. Compute and print:
#        - Total time, throughput (req/s)
#        - p50, p95, p99 latencies
#        - Error count
#     7. Plot:
#        - Latency distribution histogram
#        - Latency over time (request index vs latency)
#     8. Return results dict

# YOUR CODE HERE


# --- Commented Solution ---
# def load_test(client, n_requests=200, n_workers=10):
#     """Run a load test against the prediction endpoint."""
#     # Reset stats
#     request_stats['total_requests'] = 0
#     request_stats['total_latency'] = 0.0
#     request_stats['latencies'] = []
#
#     payload = {
#         "tenure_months": 24,
#         "monthly_charges": 65.50,
#         "contract": "One year",
#         "internet_service": "Fiber optic",
#         "online_security": 1,
#         "tech_support": 0,
#         "total_charges": 1572.00
#     }
#
#     latencies = []
#     errors = 0
#
#     def send_request():
#         start = time.time()
#         response = client.post("/predict", json=payload)
#         latency = (time.time() - start) * 1000  # ms
#         return latency, response.status_code
#
#     print(f"Starting load test: {n_requests} requests, {n_workers} workers...")
#     overall_start = time.time()
#
#     with ThreadPoolExecutor(max_workers=n_workers) as executor:
#         futures = [executor.submit(send_request) for _ in range(n_requests)]
#         for future in as_completed(futures):
#             latency, status_code = future.result()
#             latencies.append(latency)
#             if status_code != 200:
#                 errors += 1
#
#     overall_time = time.time() - overall_start
#     throughput = n_requests / overall_time
#
#     # Compute percentiles
#     p50 = np.percentile(latencies, 50)
#     p95 = np.percentile(latencies, 95)
#     p99 = np.percentile(latencies, 99)
#
#     print(f"\nLoad Test Results:")
#     print(f"  Total time: {overall_time:.2f}s")
#     print(f"  Throughput: {throughput:.1f} req/s")
#     print(f"  p50 latency: {p50:.2f}ms")
#     print(f"  p95 latency: {p95:.2f}ms")
#     print(f"  p99 latency: {p99:.2f}ms")
#     print(f"  Errors: {errors}/{n_requests}")
#
#     # Plot results
#     fig, axes = plt.subplots(1, 2, figsize=(14, 5))
#
#     # Latency distribution
#     axes[0].hist(latencies, bins=40, edgecolor='black', alpha=0.7, color='steelblue')
#     axes[0].axvline(p50, color='green', linestyle='--', linewidth=2, label=f'p50={p50:.1f}ms')
#     axes[0].axvline(p95, color='orange', linestyle='--', linewidth=2, label=f'p95={p95:.1f}ms')
#     axes[0].axvline(p99, color='red', linestyle='--', linewidth=2, label=f'p99={p99:.1f}ms')
#     axes[0].set_xlabel('Latency (ms)')
#     axes[0].set_ylabel('Count')
#     axes[0].set_title('Latency Distribution')
#     axes[0].legend()
#
#     # Latency over time
#     axes[1].scatter(range(len(latencies)), latencies, alpha=0.4, s=10, color='steelblue')
#     axes[1].axhline(p50, color='green', linestyle='--', linewidth=1, label=f'p50')
#     axes[1].axhline(p95, color='orange', linestyle='--', linewidth=1, label=f'p95')
#     axes[1].set_xlabel('Request #')
#     axes[1].set_ylabel('Latency (ms)')
#     axes[1].set_title('Latency Over Time')
#     axes[1].legend()
#
#     plt.tight_layout()
#     plt.show()
#
#     return {
#         'total_time_s': overall_time,
#         'throughput_rps': throughput,
#         'p50_ms': p50,
#         'p95_ms': p95,
#         'p99_ms': p99,
#         'errors': errors,
#         'latencies': latencies
#     }

In [ ]:
# Run the load test
load_results = load_test(client, n_requests=200, n_workers=10)

In [ ]:
# Also test batch endpoint latency
batch_sizes = [1, 5, 10, 25, 50, 100]
batch_latencies = []

base_customer = {
    "tenure_months": 24,
    "monthly_charges": 65.50,
    "contract": "One year",
    "internet_service": "Fiber optic",
    "online_security": 1,
    "tech_support": 0,
    "total_charges": 1572.00
}

for size in batch_sizes:
    customers = [base_customer] * size
    start = time.time()
    response = client.post("/predict/batch", json={"customers": customers})
    latency = (time.time() - start) * 1000
    batch_latencies.append(latency)
    print(f"Batch size {size:>3}: {latency:.2f}ms ({latency/size:.2f}ms per prediction)")

# Plot batch scaling
plt.figure(figsize=(10, 5))
plt.plot(batch_sizes, batch_latencies, 'o-', linewidth=2, markersize=8, color='steelblue')
plt.xlabel('Batch Size')
plt.ylabel('Latency (ms)')
plt.title('Batch Prediction Latency Scaling')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 8: Results and Key Findings

Summarize the performance characteristics and production readiness of the service.

In [ ]:
# Summary table
print("=" * 60)
print("PERFORMANCE SUMMARY")
print("=" * 60)
print(f"{'Metric':<35} {'Value':>20}")
print("-" * 60)
print(f"{'Single prediction p50 latency':<35} {load_results['p50_ms']:>17.2f} ms")
print(f"{'Single prediction p95 latency':<35} {load_results['p95_ms']:>17.2f} ms")
print(f"{'Single prediction p99 latency':<35} {load_results['p99_ms']:>17.2f} ms")
print(f"{'Batch prediction (50 customers)':<35} {batch_latencies[4]:>17.2f} ms")
print(f"{'Throughput':<35} {load_results['throughput_rps']:>14.1f} req/s")
print(f"{'Error rate':<35} {load_results['errors']/200*100:>16.1f} %")
print(f"{'Model accuracy':<35} {MODEL_METADATA['training_accuracy']:>17.4f}")
print(f"{'Model F1 score':<35} {MODEL_METADATA['training_f1']:>17.4f}")
print("=" * 60)

In [ ]:
# TODO: Fill in your Key Findings based on the results above

key_findings = {
    "1. Single prediction p95 latency": "",        # e.g., "Under 10ms, well within SLA"
    "2. Batch vs single tradeoff": "",              # e.g., "Batch of 50 is ~Xms vs 50 single requests at ~Yms"
    "3. Error handling coverage": "",                # e.g., "Pydantic catches type errors, range errors, missing fields"
    "4. Middleware overhead": "",                    # e.g., "Negligible - adds <0.1ms per request"
    "5. Production readiness - what's missing": ""   # e.g., "Authentication, rate limiting, async DB, logging, monitoring"
}

print("KEY FINDINGS")
print("=" * 60)
for finding, answer in key_findings.items():
    print(f"\n{finding}:")
    print(f"  {answer if answer else '[TODO: Fill in your observation]'}")